# Afinament motor TAN SLPL·UV — aina-translator-es-ca
## Servei de Llengües i Política Lingüística · Universitat de València

### Configuració prèvia a Google Colab
1. Menú: **Entorn d'execució → Canvia el tipus d'entorn → GPU (T4)**
2. Munta Google Drive i puja la carpeta del corpus a `MyDrive/SLPL_TAN/corpus/`
3. Executa les cel·les en ordre

### Configuració prèvia a Kaggle
1. Activa GPU: **Settings → Accelerator → GPU T4 x2**
2. Puja els fitxers del corpus com a Dataset de Kaggle
3. Executa les cel·les en ordre

In [ ]:
import os
import sys
import pathlib

# ── Detecció robusta de l'entorn ────────────────────────────────────────────
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

IS_KAGGLE = os.path.exists('/kaggle') and not IS_COLAB

print(f'Entorn detectat: {"Google Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Local"}')

# ── Muntatge de Drive i rutes ────────────────────────────────────────────────
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CORPUS_DIR = '/content/drive/MyDrive/SLPL_TAN/corpus'
    OUTPUT_DIR = '/content/drive/MyDrive/SLPL_TAN/model-afinar'
elif IS_KAGGLE:
    CORPUS_DIR = '/kaggle/input/slpl-corpus'
    OUTPUT_DIR = '/kaggle/working/model-afinar'
else:
    CORPUS_DIR = r'C:\Users\santi\OneDrive\Documents\SLPL\taneu\corpus d entrenament i afinament'
    OUTPUT_DIR = r'C:\Users\santi\OneDrive\Documents\SLPL\taneu\model-afinar'

pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'Corpus:  {CORPUS_DIR}')
print(f'Sortida: {OUTPUT_DIR}')

In [ ]:
# Colab ja té PyTorch i CUDA. Instal·la les biblioteques addicionals.
# NOTA: executa aquesta cel·la primer i espera que acabe abans de continuar.
!pip install "transformers>=4.45" datasets sacrebleu sentencepiece evaluate accelerate --quiet

# nltk és preinstal·lat a Colab; la descàrrega del recurs és segura aquí.
import nltk
nltk.download('punkt_tab', quiet=True)

print('\u2713 Dependències instal·lades.')

In [ ]:
import torch

cuda_ok = torch.cuda.is_available()
print(f'CUDA disponible: {cuda_ok}')
if cuda_ok:
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    DEVICE = 'cuda'
    FP16   = True
    BATCH  = 16   # T4 té 16 GB → batch gran
else:
    print('\u26a0 Sense GPU. Entrenament per CPU (molt lent).')
    DEVICE = 'cpu'
    FP16   = False
    BATCH  = 4

In [ ]:
import re
import csv
import random
from pathlib import Path

# ── Funcions de càrrega ─────────────────────────────────────────────────────

def carrega_tsv(path):
    pairs = []
    with open(path, encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='\t')
        for row in reader:
            if len(row) >= 2 and row[0].strip() and row[1].strip():
                pairs.append((row[0].strip(), row[1].strip()))
    return pairs

def carrega_txt_paral_lel(path_es, path_ca):
    pairs = []
    with open(path_es, encoding='utf-8') as fe, \
         open(path_ca, encoding='utf-8') as fc:
        for s, t in zip(fe, fc):
            s, t = s.strip(), t.strip()
            if s and t:
                pairs.append((s, t))
    return pairs

def es_valid(src, tgt, min_tok=3, max_tok=120):
    s_tok = src.split()
    t_tok = tgt.split()
    if not (min_tok <= len(s_tok) <= max_tok): return False
    if not (min_tok <= len(t_tok) <= max_tok): return False
    if src == tgt: return False
    if sum(c.isalpha() for c in src) / max(len(src), 1) < 0.4: return False
    return True

# ── Càrrega de tots els fitxers ─────────────────────────────────────────────

all_pairs = []
seen = set()
corpus_path = Path(CORPUS_DIR)

# Fitxers TSV
for f in corpus_path.glob('**/*.tsv'):
    pairs = carrega_tsv(f)
    print(f'  TSV: {f.name} \u2192 {len(pairs)} parells')
    all_pairs.extend(pairs)

# Fitxers CSV
for f in corpus_path.glob('**/*.csv'):
    pairs = carrega_tsv(f)  # mateixa lògica amb coma
    print(f'  CSV: {f.name} \u2192 {len(pairs)} parells')
    all_pairs.extend(pairs)

# Fitxers TXT paral·lels (_CAS.txt / _VAL.txt)
for f_es in corpus_path.glob('**/*_CAS.txt'):
    f_ca = f_es.parent / f_es.name.replace('_CAS', '_VAL')
    if f_ca.exists():
        pairs = carrega_txt_paral_lel(f_es, f_ca)
        print(f'  TXT: {f_es.name} + {f_ca.name} \u2192 {len(pairs)} parells')
        all_pairs.extend(pairs)

# Fitxers .es / .ca
for f_es in corpus_path.glob('**/*.es'):
    f_ca = f_es.with_suffix('.ca')
    if f_ca.exists():
        pairs = carrega_txt_paral_lel(f_es, f_ca)
        print(f'  PAR: {f_es.name} + {f_ca.name} \u2192 {len(pairs)} parells')
        all_pairs.extend(pairs)

# ── Neteja i deduplicació ───────────────────────────────────────────────────

clean_pairs = []
for src, tgt in all_pairs:
    if not es_valid(src, tgt): continue
    key = f'{src.lower()}|||{tgt.lower()}'
    if key in seen: continue
    seen.add(key)
    clean_pairs.append((src, tgt))

# ── Divisió train/val ───────────────────────────────────────────────────────

random.seed(42)
random.shuffle(clean_pairs)
cut = int(len(clean_pairs) * 0.9)
train_pairs = clean_pairs[:cut]
val_pairs   = clean_pairs[cut:]

print(f'\n{"="*45}')
print(f'  CORPUS LLEST')
print(f'{"="*45}')
print(f'  Total parells vàlids: {len(clean_pairs)}')
print(f'  Entrenament:          {len(train_pairs)}')
print(f'  Validació:            {len(val_pairs)}')
print(f'{"="*45}')

if len(clean_pairs) < 500:
    print('\n\u26a0 Corpus molt petit (< 500 parells).')
    print('  Recomana acumular més textos abans de fer l\'afinament.')

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

MODEL_ID = 'projecte-aina/aina-translator-es-ca'
print(f'Carregant model: {MODEL_ID}')

tokenizer = MarianTokenizer.from_pretrained(MODEL_ID)
model     = MarianMTModel.from_pretrained(MODEL_ID)

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'\u2713 Model carregat. Paràmetres: {params:.1f}M')

# Prova de traducció base (abans de l'afinament)
test = 'El servicio de lenguas ofrece asesoramiento lingüístico a la comunidad universitaria.'
inputs = tokenizer([test], return_tensors='pt', padding=True)
output = model.generate(**inputs)
print(f'\nProva base:')
print(f'  ES: {test}')
print(f'  CA: {tokenizer.decode(output[0], skip_special_tokens=True)}')

In [ ]:
from datasets import Dataset

def preprocess(examples):
    sources = [ex['es'] for ex in examples['translation']]
    targets = [ex['ca'] for ex in examples['translation']]
    model_inputs = tokenizer(
        sources, max_length=128, truncation=True, padding='max_length'
    )
    # text_target= substitueix l'API deprecada as_target_tokenizer()
    # (vàlid des de transformers 4.21+)
    labels = tokenizer(
        text_target=targets,
        max_length=128,
        truncation=True,
        padding='max_length'
    )
    model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels['input_ids']
    ]
    return model_inputs

train_dataset = Dataset.from_dict({'translation': [{'es': s, 'ca': t} for s, t in train_pairs]})
val_dataset   = Dataset.from_dict({'translation': [{'es': s, 'ca': t} for s, t in val_pairs]})

print('Tokenitzant corpus...', end=' ')
train_tok = train_dataset.map(preprocess, batched=True, num_proc=2, remove_columns=['translation'])
val_tok   = val_dataset.map(preprocess,   batched=True, num_proc=2, remove_columns=['translation'])
train_tok.set_format('torch')
val_tok.set_format('torch')
print('fet.')
print(f'\u2713 Datasets preparats: {len(train_tok)} train | {len(val_tok)} val')

In [ ]:
import evaluate
import numpy as np
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

metric = evaluate.load('sacrebleu')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = metric.compute(
        predictions=[p.strip() for p in decoded_preds],
        references=[[l.strip()] for l in decoded_labels]
    )
    return {'bleu': round(result['score'], 2)}

training_args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = BATCH,
    per_device_eval_batch_size  = BATCH,
    warmup_steps                = 200,
    weight_decay                = 0.01,
    learning_rate               = 5e-5,
    fp16                        = FP16,
    predict_with_generate       = True,
    generation_max_length       = 128,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'bleu',
    logging_steps               = 50,
    report_to                   = 'none',
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=train_tok, eval_dataset=val_tok,
    tokenizer=tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f'Iniciant afinament: {training_args.num_train_epochs} epochs')
print(f'Corpus: {len(train_tok)} frases train | {len(val_tok)} val')
print(f'Batch size: {BATCH} | FP16: {FP16} | Device: {DEVICE}')
print('\u2500' * 50)

train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f'\n\u2713 Afinament completat.')
print(f'  Pèrdua final: {train_result.training_loss:.4f}')
print(f'  Model guardat a: {OUTPUT_DIR}')

In [ ]:
import sacrebleu as sb
from transformers import pipeline

translator = pipeline(
    'translation',
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=0 if DEVICE == 'cuda' else -1,
    max_length=128
)

val_src = [p[0] for p in val_pairs]
val_ref = [p[1] for p in val_pairs]

print('Generant traduccions de validació...', end=' ')
translations = [t['translation_text'] for t in translator(val_src, batch_size=32)]
print('fet.')

bleu = sb.corpus_bleu(translations, [val_ref])
chrf = sb.corpus_chrf(translations, [val_ref])

print(f'\n{"="*45}')
print(f'  RESULTATS FINALS')
print(f'{"="*45}')
print(f'  BLEU:  {bleu.score:.2f}')
print(f'  ChrF:  {chrf.score:.2f}')
print(f'{"="*45}')

print('\nExemples de traducció del model afinar:')
for i in range(min(5, len(val_src))):
    print(f'\n  [{i+1}] ES:  {val_src[i]}')
    print(f'       CA:  {translations[i]}')
    print(f'  REF:      {val_ref[i]}')

In [ ]:
# Descarrega el model afinar al teu ordinador
import zipfile, os

zip_name = '/content/model_afinar_SLPL_UV.zip' if IS_COLAB else '/kaggle/working/model_afinar_SLPL_UV.zip'

print(f'Comprimint el model...', end=' ')
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file_path in pathlib.Path(OUTPUT_DIR).rglob('*'):
        if file_path.is_file():
            zf.write(file_path, file_path.relative_to(OUTPUT_DIR))

size_mb = os.path.getsize(zip_name) / 1024**2
print(f'fet. ({size_mb:.0f} MB)')

if IS_COLAB:
    from google.colab import files
    print('Iniciant descàrrega...')
    files.download(zip_name)
else:
    print(f'Model disponible a: {zip_name}')
    print('Descarrega\'l des del panell de fitxers de Kaggle.')

print(f'\nDesprés de descomprimir, copia la carpeta a:')
print(f'  C:\\Users\\santi\\OneDrive\\Documents\\SLPL\\taneu\\model-afinar\\')